# Qwen2.5-Coder-7B-Instruct: полные SQL-запросы на H&M small-v1

Новый эксперимент: Qwen возвращает **обычный SQL**, не JSON aggregates/derived.
Она пишет один или несколько полных SELECT/WITH-запросов: группировки, CTE,
формулы, aliases и при необходимости окна. Наш код проверяет ключи/числовые
признаки, отсутствие дубликатов ключей и присоединяет результаты к кандидатам.
Модель видит только физически отфильтрованную прошлую history; будущих транзакций,
целей и произвольных таблиц в её SQL-соединении нет. Изменение интерфейса —
осознанная адаптация, не точное повторение старого DSL или статьи.

1. Импортируйте этот `.ipynb` в свежую Kaggle-сессию.
2. Включите **Internet** и **GPU T4**. Qwen публична, HF_TOKEN не нужен.
3. Добавьте прежний **hm_small_sql_compare_v1.zip** через Dataset/Input.
4. **Save Version → Run All**. Старые отчёты не нужны как Input.
5. Скачайте `hm_small_qwen7b_full_sql_v2_<hash>_results.zip` из Output.

Один поиск:20 различных успешно оценённых наборов, до300 обращений,
до3 исправлений,3 популяции. Полные SQL-наборы сравниваются целиком.
Сохраняются все ответы и `program.sql`, ошибки, популяции/repair/parent,
дубликаты, модели, прогнозы, метрики, графики поиска/logloss/сравнения и AP-гистограммы.
ZIP обновляется после каждого вызова. Cancel Kaggle не гарантирует публикацию Output.

Версия2: исправлен ложный отказ логических AND/OR/NOT/IN/BETWEEN/IS.
Все остальные данные/параметры/prompt сохранены, старый v1 не изменён.


## Условия сравнения

Те же72813 train-пар/4804 user-time/8cutoff,raw15,каталог5000 и val304/test250.
LightGBM150rounds/31leaves/4CPUthreads;NF4/doublequant/FP16 наT4,temperature0.8,
input6144/output2048,seed42+call−1. Та же закреплённая revision Qwen7B.
Search128 → top3+raw → confirmation88;heldout88 отдельно. Выбор фиксируется доtest.
Test уже просматривался: повторная диагностическая оценка, не новый закрытый test.
Старые Qwen7B/H003 результаты сравним локально после получения ZIP.

Новый интерфейс:до6 независимых SELECT-запросов,до36 числовых f_* выходов всего.
Нет разделения на24 aggregates/12 derived: отношения/формулы вычисляет SQL.
Поддержаны CTE,aliases,внутренние joins и числовые окна при уникальности конечных ключей.
Дедупликация консервативная по SQL AST: пробелы/регистр/комментарии и переименование f_*
нормализуются; общая алгебраическая эквивалентность/переименование CTE не распознаются.
Поэтому эксперимент меняет также пространство программ и критерий дубликата.

SQL-парсер:[SQLGlot](https://sqlglot.com/),закреплена
[версия27.20.0](https://pypi.org/project/sqlglot/27.20.0/).
LightGBM,код обучения/оценки,raw-кодирование и конечные joins сохранены прежними.


In [ ]:
import subprocess, sys
# Сохраняем предустановленные torch/CUDA. Запускайте в свежей Kaggle-сессии.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1',
    'sqlglot==27.20.0', 'lightgbm==4.6.0', 'duckdb==1.4.1', 'sentencepiece==0.2.1', 'pyarrow>=18'])


In [ ]:
from pathlib import Path
import os, json, hashlib, time, traceback, importlib.metadata, gc, shutil
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

ARMS = ['qwen_full_sql']
MODELS = {'qwen': {'model_id': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'revision': 'c03e6d358207e414f1eca0bb1891e29f1db0e242'}}
CONFIG = {
    'protocol': 'hm_small_qwen7b_full_sql_v2', 'arms': ARMS, 'test_previously_inspected': True, 'seed': 42,
    'target_successful_distinct': 20, 'max_llm_calls': 300, 'max_repairs_per_proposal': 3,
    'islands': 3, 'memory_programs_per_prompt': 1,
    'temperature': .8, 'max_input_tokens': 6144, 'max_new_tokens': 2048,
    'lightgbm_rounds': 150, 'cpu_threads': 4, 'scoring_user_batch': 16,
    'evaluation_timeout_seconds': 1800,
    'quantization': 'bnb NF4/double_quant/FP16',
    'feature_interface': 'complete_SQL_SELECT_queries_with_host_joins',
    'search_users': 128, 'confirmation_users': 88, 'heldout_val_users': 88,
    'validation_users': 304, 'test_users': 250, 'evaluation_catalog': 5000,
    'hypothesis': 'H004 full SELECT scripts', 'max_sql_queries': 6, 'max_sql_features': 36,
}
WORK = Path('/kaggle/working')
# Download caches excluded from Output; remove dedicated model cache after unloading to save disk.
CACHE_ROOT = Path('/tmp/hm_small_qwen7b_full_sql_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
MODULES = {'base_features.py': '"""Compile LLM-authored SQL feature programs over time-filtered H&M history.\n\nOnly scalar/aggregate expressions are accepted, never executable Python or SQL\nstatements. Joins, labels, temporal boundaries and row order belong to the host.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\n\nimport duckdb\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nROOT = Path(__file__).resolve().parents[1]\nITEM_COLUMNS = ["product_code", "product_type_no", "graphical_appearance_no",\n                "colour_group_code", "perceived_colour_value_id", "perceived_colour_master_id",\n                "department_no", "index_group_no", "section_no", "garment_group_no"]\nUSER_COLUMNS = ["age", "FN", "Active", "club_member_status", "fashion_news_frequency"]\nGROUP_COLUMNS = ["customer_id", "article_id", *ITEM_COLUMNS]\nFUNCTIONS = {"count", "sum", "avg", "min", "max", "stddev_pop", "stddev_samp",\n             "variance", "median", "quantile_cont", "approx_count_distinct", "mode",\n             "coalesce", "nullif", "abs", "sqrt", "ln", "log", "log2", "log10", "exp",\n             "pow", "power", "greatest", "least", "round", "floor", "ceil", "cast",\n             "filter", "in", "not", "where", "when", "then", "else", "and", "or"}\nFORBIDDEN = re.compile(r"\\b(select|from|join|union|with|over|attach|detach|copy|pragma|install|load|"\n                       r"create|drop|alter|insert|update|delete|read_csv|read_parquet|query|"\n                       r"customer_id|article_id|rid|label|target|timestamp|t_dat)\\b", re.I)\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef validate_expression(expression, aggregate=False):\n    if not isinstance(expression, str) or len(expression) > 1000:\n        raise ValueError("SQL expression must be a string of <=1000 characters")\n    if any(part in expression for part in [";", "--", "/*", "*/", "\'", \'"\', "\\\\", "."]):\n        # Decimal literals are permitted, qualified identifiers and strings are not.\n        stripped = re.sub(r"\\b\\d+\\.\\d+\\b", "0", expression)\n        if any(part in stripped for part in [";", "--", "/*", "*/", "\'", \'"\', "\\\\", "."]):\n            raise ValueError("Only numeric SQL expressions, no statements/strings/qualified names")\n    checked_expression = expression\n    if aggregate:\n        checked_expression = re.sub(\n            r"\\b(?:count\\s*\\(\\s*(?:distinct\\s+)?|approx_count_distinct\\s*\\(\\s*)(?:customer_id|article_id)\\s*\\)",\n            "count(*)", checked_expression, flags=re.I)\n    forbidden = FORBIDDEN.search(checked_expression)\n    if forbidden:\n        raise ValueError(f"Forbidden token {forbidden.group(0)!r}: IDs only in grouping keys/counts; no SQL statements, window functions, or labels")\n    function_expression = expression\n    if os.environ.get("LLMFE_VALIDATOR_VERSION") == "h002":\n        # WITHIN GROUP (...) is SQL syntax, not a call to a function GROUP.\n        function_expression = re.sub(r"\\bWITHIN\\s+GROUP\\s*\\(", "(", expression, flags=re.I)\n    for function in re.findall(r"\\b([a-zA-Z_][a-zA-Z_0-9]*)\\s*\\(", function_expression):\n        if function.lower() not in FUNCTIONS:\n            raise ValueError(f"Function outside allowlist: {function}")\n    return expression\n\n\ndef validate_program(program):\n    if set(program) - {"rationale", "aggregates", "derived"}:\n        raise ValueError("Unknown program keys")\n    aggs, derived = program.get("aggregates", []), program.get("derived", [])\n    if not isinstance(aggs, list) or not isinstance(derived, list) or len(aggs) > 24 or len(derived) > 12:\n        raise ValueError("Maximum 24 aggregates and 12 derived features")\n    names = set()\n    for entry in aggs + derived:\n        if set(entry) - {"name", "keys", "sql"}:\n            raise ValueError("Unknown feature keys")\n        name = entry["name"]\n        if not re.fullmatch(r"f_[a-z][a-z0-9_]{0,45}", name) or name in names:\n            raise ValueError("Feature names must be unique f_lowercase_names")\n        names.add(name)\n        validate_expression(entry["sql"], aggregate=entry in aggs)\n    for entry in aggs:\n        keys = entry["keys"]\n        if (not isinstance(keys, list) or not keys or len(keys) > 2 or len(set(keys)) != len(keys)\n                or any(key not in GROUP_COLUMNS for key in keys)):\n            raise ValueError("Invalid grouping keys")\n        if "customer_id" not in keys and len(keys) != 1:\n            raise ValueError("Use one article/category key, customer_id, or customer_id + article/category")\n    return program\n\n\nclass FeatureEngine:\n    def __init__(self, scratch, data_dir=None):\n        self.scratch = Path(scratch)\n        self.scratch.mkdir(parents=True, exist_ok=True)\n        self.data_dir = Path(data_dir or ROOT / "data/rel-hm/db")\n        self.con = duckdb.connect()\n        self.con.execute("SET threads=2")\n        self.con.execute("SET memory_limit=\'2300MB\'")\n        spill = str(self.scratch / "spill").replace("\'", "\'\'")\n        self.con.execute(f"SET temp_directory=\'{spill}\'")\n        self.con.execute("SET max_temp_directory_size=\'6GB\'")\n        self.items = pd.read_parquet(self.data_dir / "article.parquet", columns=["article_id", *ITEM_COLUMNS])\n        self.users = pd.read_parquet(self.data_dir / "customer.parquet", columns=["customer_id", *USER_COLUMNS])\n        self.category_maps = {}\n        for column in ITEM_COLUMNS:\n            encoded, categories = pd.factorize(self.items[column], sort=True)\n            self.category_maps[column] = categories.tolist()\n            self.items[column] = encoded.astype(np.int32)\n        for column in ["club_member_status", "fashion_news_frequency"]:\n            values = self.users[column].astype("category")\n            self.category_maps[column] = list(values.cat.categories.astype(str))\n            self.users[column] = values.cat.codes.astype(np.float32)\n        # This is raw categorical encoding, not historical manual features.\n        for name, frame in [("articles", self.items), ("customers", self.users)]:\n            self.con.register("input_frame", frame)\n            self.con.execute(f"CREATE TABLE {name} AS SELECT * FROM input_frame")\n            self.con.unregister("input_frame")\n        raw_path = str(self.data_dir / "transactions.parquet").replace("\'", "\'\'")\n        self.con.execute(f"CREATE TABLE tx AS SELECT * FROM read_parquet(\'{raw_path}\')")\n        self.raw_names = [f"raw_u_{col.lower()}" for col in USER_COLUMNS] + [f"raw_i_{col}" for col in ITEM_COLUMNS]\n        self.program = None\n\n    def prepare(self, timestamp, users, program):\n        self.program = validate_program(program)\n        wanted = pd.DataFrame({"customer_id": np.unique(np.asarray(users, dtype=np.int64))})\n        self.con.register("wanted_input", wanted)\n        self.con.execute("CREATE OR REPLACE TEMP TABLE wanted AS SELECT * FROM wanted_input")\n        self.con.unregister("wanted_input")\n        cutoff = pd.Timestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S")\n        self.con.execute(f"""CREATE OR REPLACE TEMP VIEW history AS\n            SELECT t.customer_id,t.article_id,t.price,t.sales_channel_id,\n            date_diff(\'day\',t.t_dat,TIMESTAMP \'{cutoff}\')::INTEGER AS days_ago,\n            a.* EXCLUDE(article_id), c.age AS customer_age\n            FROM tx t LEFT JOIN articles a USING(article_id)\n            LEFT JOIN customers c USING(customer_id)\n            WHERE t.t_dat <= TIMESTAMP \'{cutoff}\'""")\n        groups = {}\n        for entry in program.get("aggregates", []):\n            groups.setdefault(tuple(entry["keys"]), []).append(entry)\n        self.groups = []\n        for index, (keys, entries) in enumerate(groups.items()):\n            table = f"aggregate_{index}"\n            feature_sql = ",".join(f"{entry[\'sql\']} AS {entry[\'name\']}" for entry in entries)\n            source = "history JOIN wanted USING(customer_id)" if "customer_id" in keys else "history"\n            columns = ",".join(keys)\n            self.con.execute(f"CREATE OR REPLACE TEMP TABLE {table} AS SELECT {columns},{feature_sql} FROM {source} GROUP BY {columns}")\n            self.groups.append((table, keys, entries))\n\n    @property\n    def feature_names(self):\n        return self.raw_names + [entry["name"] for entry in self.program.get("aggregates", [])] + [entry["name"] for entry in self.program.get("derived", [])]\n\n    def frame(self, pairs):\n        # The feature SQL never receives labels, target lists or arbitrary files.\n        inputs = pairs[["customer_id", "article_id"]].copy().reset_index(drop=True)\n        inputs.insert(0, "rid", np.arange(len(inputs), dtype=np.int64))\n        self.con.register("input_pairs", inputs)\n        raw = [f\'c."{col}" AS raw_u_{col.lower()}\' for col in USER_COLUMNS]\n        raw += [f"a.{col} AS raw_i_{col}" for col in ITEM_COLUMNS]\n        joins, aggregate_select = [], []\n        for table, keys, entries in self.groups:\n            conditions = []\n            for key in keys:\n                owner = "p" if key in ["customer_id", "article_id"] else "a"\n                conditions.append(f"{table}.{key}={owner}.{key}")\n            joins.append(f"LEFT JOIN {table} ON {\' AND \'.join(conditions)}")\n            aggregate_select.extend(f"{table}.{entry[\'name\']} AS {entry[\'name\']}" for entry in entries)\n        select = ",".join(["p.rid", *raw, *aggregate_select])\n        query = f"SELECT {select} FROM input_pairs p LEFT JOIN customers c USING(customer_id) LEFT JOIN articles a USING(article_id) {\' \'.join(joins)}"\n        # Derived expressions see only raw/aggregate feature columns. Each stage\n        # may reference earlier derived features, but cannot access row IDs.\n        for entry in self.program.get("derived", []):\n            query = f"SELECT *, {entry[\'sql\']} AS {entry[\'name\']} FROM ({query})"\n        ordered = ",".join(self.feature_names)\n        try:\n            frame = self.con.sql(f"SELECT {ordered} FROM ({query}) ORDER BY rid").df()\n        finally:\n            self.con.unregister("input_pairs")\n        if len(frame) != len(inputs) or list(frame) != self.feature_names:\n            raise RuntimeError("Feature rows/schema changed")\n        for column in frame:\n            if not pd.api.types.is_numeric_dtype(frame[column]):\n                raise ValueError(f"Non-numeric feature: {column}")\n        return frame\n\n    def matrix(self, pairs):\n        values = self.frame(pairs).to_numpy(dtype=np.float32)\n        return np.nan_to_num(values, nan=-1., posinf=-1., neginf=-1.)\n\n    def close(self):\n        self.con.close()\n', 'worker.py': '"""Isolated CPU evaluation using the original local Qwen SQL FeatureEngine."""\nimport os\nif __name__ == \'__main__\':\n    os.environ.update(CUDA_VISIBLE_DEVICES=\'\', OMP_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\')\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport signal\nimport time\nimport numpy as np\nimport pandas as pd\nimport lightgbm as lgb\nfrom features import FeatureEngine, validate_program\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write(path, value):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, indent=2), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef ap12(pred, targets):\n    pred = np.asarray(pred)\n    assert pred.shape == (len(targets), 12)\n    assert all(len(set(row)) == 12 for row in pred)\n    hits = np.array([np.isin(row, np.unique(t)) for row, t in zip(pred, targets)])\n    denominators = np.minimum(12, [len(np.unique(t)) for t in targets])\n    assert (denominators > 0).all()\n    return ((hits.cumsum(1) / np.arange(1, 13)) * hits).sum(1) / denominators\n\n\ndef timed_frame(engine, pairs):\n    def timeout(*_):\n        raise TimeoutError(\'Feature construction exceeded 60 seconds\')\n    previous = signal.signal(signal.SIGALRM, timeout)\n    signal.alarm(60)\n    try:\n        return engine.matrix(pairs)\n    finally:\n        signal.alarm(0)\n        signal.signal(signal.SIGALRM, previous)\n\n\ndef preflight(engine, pairs, program):\n    for cutoff in sorted(pairs.timestamp.unique())[:1] + sorted(pairs.timestamp.unique())[-1:]:\n        sample = pairs[pairs.timestamp == cutoff].head(128).reset_index(drop=True)\n        engine.prepare(cutoff, sample.customer_id.values, program)\n        whole = timed_frame(engine, sample)\n        permutation = np.random.default_rng(42).permutation(len(sample))\n        permuted = timed_frame(engine, sample.iloc[permutation].reset_index(drop=True))\n        np.testing.assert_allclose(permuted[np.argsort(permutation)], whole, equal_nan=True)\n        middle = len(sample) // 2\n        chunks = np.concatenate([timed_frame(engine, sample.iloc[:middle]), timed_frame(engine, sample.iloc[middle:])])\n        np.testing.assert_allclose(chunks, whole, equal_nan=True)\n    return {\'passed\': True, \'row_permutation\': True, \'batch_invariance\': True}\n\n\ndef fit(engine, root, directory, program, config):\n    pairs = pd.read_parquet(root / \'train_pairs.parquet\')\n    assert len(pairs) == config[\'train_pairs\']\n    checks = preflight(engine, pairs, program)\n    parts = []\n    labels = []\n    for cutoff, frame in pairs.groupby(\'timestamp\', sort=True):\n        engine.prepare(cutoff, frame.customer_id.values, program)\n        parts.append(timed_frame(engine, frame))\n        labels.append(frame.label.to_numpy())\n    values, y = np.concatenate(parts), np.concatenate(labels)\n    params = dict(objective=\'binary\', verbosity=-1, learning_rate=.05, num_leaves=31,\n                  min_data_in_leaf=100, max_bin=127, feature_fraction=.9, bagging_fraction=.8,\n                  bagging_freq=1, deterministic=True, force_col_wise=True, seed=42,\n                  feature_fraction_seed=42, bagging_seed=42, data_random_seed=42,\n                  num_threads=config[\'cpu_threads\'])\n    # Same fixed downstream settings for both LLMs; reference manual model tuned leaves15/31.\n    names = engine.feature_names\n    categorical = [n for n in names if n.startswith(\'raw_\') and n not in {\'raw_u_age\', \'raw_u_fn\', \'raw_u_active\'}]\n    dataset = lgb.Dataset(values, label=y, feature_name=names, categorical_feature=categorical)\n    history = {}\n    model = lgb.train(params, dataset, num_boost_round=config[\'lightgbm_rounds\'], valid_sets=[dataset],\n                      valid_names=[\'train\'], callbacks=[lgb.record_evaluation(history)])\n    model.save_model(str(directory / \'model.txt\'))\n    write(directory / \'learning_curve.json\', history)\n    write(directory / \'feature_schema.json\', {\'features\': names, \'categorical\': categorical,\n          \'train_pairs\': len(y), \'train_pair_sha256\': config[\'train_pair_sha256\'], \'preflight\': checks,\n          \'category_maps\': engine.category_maps, \'params\': params})\n    del values, y, parts, dataset, model\n    gc.collect()\n\n\ndef predict(engine, root, directory, stage, program, config, output_stage=None):\n    split = \'test\' if stage == \'test\' else \'val\'\n    target = pd.read_parquet(root / f\'{split}_targets.parquet\')\n    if stage in {\'search\', \'confirm\', \'heldout_val\'}:\n        selected = json.loads((root / \'selection.json\').read_text())[stage]\n        target = target.iloc[selected].reset_index(drop=True)\n    catalog = np.sort(engine.items.article_id.to_numpy(dtype=np.int64))\n    assert np.array_equal(catalog, np.arange(5000))\n    assert target.timestamp.nunique() == 1\n    engine.prepare(target.timestamp.iloc[0], target.customer_id.values, program)\n    model = lgb.Booster(model_file=str(directory / \'model.txt\'))\n    all_predictions = []\n    for start in range(0, len(target), config[\'scoring_user_batch\']):\n        users = target.customer_id.iloc[start:start + config[\'scoring_user_batch\']].to_numpy()\n        pairs = pd.DataFrame({\'customer_id\': np.repeat(users, len(catalog)), \'article_id\': np.tile(catalog, len(users))})\n        scores = model.predict(timed_frame(engine, pairs), num_threads=config[\'cpu_threads\']).reshape(len(users), len(catalog))\n        assert np.isfinite(scores).all()\n        all_predictions.extend([catalog[np.lexsort((catalog, -row))[:12]] for row in scores])\n    pred = np.array(all_predictions)\n    ap = ap12(pred, target.article_id)\n    output_stage = output_stage or stage\n    np.save(directory / f\'{output_stage}_predictions.npy\', pred)\n    np.save(directory / f\'{output_stage}_ap.npy\', ap)\n    np.save(directory / f\'{output_stage}_users.npy\', target.customer_id.to_numpy())\n    return {\'map\': float(ap.mean()), \'users\': len(target), \'pairs\': len(target) * len(catalog),\n            \'full_catalog\': 5000, \'predictions_sha256\': sha(directory / f\'{output_stage}_predictions.npy\')}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    for name in [\'data\', \'directory\', \'stage\', \'config\']:\n        parser.add_argument(\'--\' + name, required=True)\n    args = parser.parse_args()\n    root, directory = Path(args.data), Path(args.directory)\n    config = json.loads(Path(args.config).read_text())\n    source = directory / \'program.json\'\n    program = validate_program(json.loads(source.read_text()))\n    started = time.perf_counter()\n    engine = FeatureEngine(directory / \'scratch\', root)\n    engine.con.execute(\'SET enable_external_access=false\')\n    try:\n        if args.stage == \'search\' and not (directory / \'model.txt\').exists():\n            fit(engine, root, directory, program, config)\n        elif args.stage == \'preflight\':\n            checks = preflight(engine, pd.read_parquet(root / \'train_pairs.parquet\'), program)\n            write(directory / \'preflight_metrics.json\', checks)\n            return\n        stage = args.stage.removeprefix(\'replay_\')\n        metric = predict(engine, root, directory, stage, program, config, output_stage=args.stage)\n        metric.update(program_sha256=sha(source), model_sha256=sha(directory / \'model.txt\'), seconds=time.perf_counter() - started)\n        if args.stage.startswith(\'replay_\'):\n            # Save replay separately; original files are preserved by host before running replay.\n            original = json.loads((directory / f\'{stage}_metrics.json\').read_text())\n            assert metric[\'predictions_sha256\'] == original[\'predictions_sha256\'], \'Frozen model replay differs\'\n            assert metric[\'map\'] == original[\'map\']\n            metric[\'passed\'] = True\n        write(directory / f\'{args.stage}_metrics.json\', metric)\n    finally:\n        engine.close()\n\n\nif __name__ == \'__main__\':\n    main()\n', 'features.py': '"""Full SELECT interface; frozen host raw encoding and joins in base_features."""\nfrom full_sql import FeatureEngine, validate_program\n', 'full_sql.py': '"""Read-only full SELECT programs over a physically past-only history database."""\nimport hashlib\nimport json\nimport re\nimport threading\nimport duckdb\nimport numpy as np\nimport pandas as pd\nimport sqlglot\nfrom sqlglot import exp\nfrom base_features import FeatureEngine as BaseEngine, ITEM_COLUMNS, GROUP_COLUMNS\n\nHISTORY = [\'customer_id\', \'article_id\', \'price\', \'sales_channel_id\', \'days_ago\', \'customer_age\', *ITEM_COLUMNS]\nMAX_QUERIES = 6\nMAX_FEATURES = 36\nFUNCTIONS = {\'COUNT\',\'SUM\',\'AVG\',\'MIN\',\'MAX\',\'STDDEV_POP\',\'STDDEV_SAMP\',\'VARIANCE\',\'VAR_POP\',\'VAR_SAMP\',\n             \'MEDIAN\',\'QUANTILE_CONT\',\'PERCENTILE_CONT\',\'APPROX_DISTINCT\',\'APPROX_COUNT_DISTINCT\',\'MODE\',\n             \'COALESCE\',\'NULLIF\',\'ABS\',\'SQRT\',\'LN\',\'LOG\',\'LOG2\',\'LOG10\',\'EXP\',\'POW\',\'POWER\',\'GREATEST\',\n             \'LEAST\',\'ROUND\',\'FLOOR\',\'CEIL\',\'CEILING\',\'CAST\',\'TRY_CAST\',\'CASE\',\'IF\',\'ROW_NUMBER\',\'RANK\',\n             \'DENSE_RANK\',\'LAG\',\'LEAD\',\'FIRST_VALUE\',\'LAST_VALUE\',\'NTH_VALUE\',\'NTILE\'}\n# SQLGlot models standard Boolean syntax as Func nodes. These are not external functions.\nFUNCTIONS.update({\'AND\',\'OR\',\'NOT\',\'IN\',\'BETWEEN\',\'IS\'})\nNUMERIC_TYPE = re.compile(r\'(BOOLEAN|TINYINT|SMALLINT|INTEGER|BIGINT|HUGEINT|UTINYINT|USMALLINT|UINTEGER|UBIGINT|UHUGEINT|FLOAT|DOUBLE|DECIMAL\\(\\d+,\\s*\\d+\\))\')\n\n\nclass SQLProgramError(ValueError):\n    def __init__(self, issues):\n        self.issues = issues\n        super().__init__(\'Fix ALL listed queries; return the COMPLETE SQL script.\\n\' + \'\\n\'.join(\n            f"query{e[\'query\']}: {e[\'reason\'][:240]} | {e[\'hint\'][:120]}" for e in issues))\n\n\ndef extract_sql(text):\n    if not isinstance(text,str) or not text.strip():\n        raise ValueError(\'Return a complete SELECT/WITH SQL script, not JSON/Python\')\n    if \'```\' in text:\n        matches = list(re.finditer(r\'```(?:sql|duckdb)?\\s*\\n(.*?)```\',text,re.S|re.I))\n        if not matches or re.sub(r\'```(?:sql|duckdb)?\\s*\\n.*?```\',\'\',text,flags=re.S|re.I).strip():\n            raise ValueError(\'Return only SQL, optionally in SQL code fences; no prose/JSON\')\n        text=\';\\n\'.join(m.group(1).strip().rstrip(\';\') for m in matches)\n    if len(text)>24000:\n        raise ValueError(\'SQL script too long; use <=6 queries and <=36 total numeric features\')\n    return text.strip()\n\n\ndef validate_ast(query):\n    nodes=sqlglot.parse(query,read=\'duckdb\')\n    if len(nodes)!=1 or not isinstance(nodes[0],exp.Select):\n        raise ValueError(\'Each statement must be SELECT or WITH ... SELECT; no DDL/DML/UNION/commands\')\n    tree=nodes[0]\n    if any(isinstance(n,(exp.Insert,exp.Update,exp.Delete,exp.Create,exp.Drop,exp.Command,exp.Into)) for n in tree.walk()):\n        raise ValueError(\'Only read-only SELECT queries are supported\')\n    if any(isinstance(n,exp.With) and n.args.get(\'recursive\') for n in tree.walk()):\n        raise ValueError(\'Recursive CTEs are not supported\')\n    if any(isinstance(n,exp.TableSample) for n in tree.walk()):\n        raise ValueError(\'Random table sampling is not supported; use deterministic history statistics\')\n    ctes={n.alias_or_name.lower() for n in tree.find_all(exp.CTE)}\n    if ctes.intersection({\'history\',\'tx\',\'articles\',\'customers\',\'input_pairs\'}):\n        raise ValueError(\'CTE names must not shadow host tables; choose e.g. counts or recent\')\n    for table in tree.find_all(exp.Table):\n        if not isinstance(table.this,exp.Identifier) or table.db or table.catalog:\n            raise ValueError(\'No table functions, files, qualified databases, or external sources\')\n        if table.name.lower() not in {\'history\',*ctes}:\n            raise ValueError(\'Unknown table \'+table.name+\'; only history and your own CTEs are available\')\n    if not any(t.name.lower()==\'history\' for t in tree.find_all(exp.Table)):\n        raise ValueError(\'Each query must read past-only history, directly or through its CTEs\')\n    for function in tree.find_all(exp.Func):\n        name=function.name.upper() if isinstance(function,exp.Anonymous) else function.sql_name().upper()\n        if name not in FUNCTIONS:\n            raise ValueError(\'Function outside supported numeric SQL operations: \'+name)\n    for alias in tree.find_all(exp.Alias):\n        if alias.alias.lower() in GROUP_COLUMNS:\n            if not isinstance(alias.this,exp.Column) or alias.this.name.lower()!=alias.alias.lower():\n                raise ValueError(\'Join keys must retain original values/names; do not transform \'+alias.alias)\n    # IDs are valid keys/count inputs, never numeric predictors or arithmetic operands.\n    for column in tree.find_all(exp.Column):\n        if column.name.lower() not in {\'customer_id\',\'article_id\'}:\n            continue\n        parent=column.parent\n        if isinstance(parent,exp.Select) and column in parent.expressions: continue\n        if isinstance(parent,exp.Alias) and parent.alias.lower()==column.name.lower(): continue\n        if isinstance(parent,exp.Group): continue\n        if column.find_ancestor(exp.Count,exp.ApproxDistinct) is not None: continue\n        if isinstance(parent,exp.Window) and column in parent.args.get(\'partition_by\',[]): continue\n        if isinstance(parent,exp.EQ) and isinstance(parent.left,exp.Column) and isinstance(parent.right,exp.Column):\n            if parent.left.name.lower()==parent.right.name.lower(): continue\n        raise ValueError(\'IDs are join/group keys or count inputs, not feature values: \'+column.sql())\n    return tree\n\n\ndef schema(con,query):\n    con.execute(\'EXPLAIN \'+query)\n    description=con.execute(\'DESCRIBE \'+query).fetchall()\n    names=[d[0] for d in description]\n    if len(names)!=len(set(names)):\n        raise ValueError(\'Duplicate output column names\')\n    keys=[n for n in names if n in GROUP_COLUMNS]\n    features=[n for n in names if n not in keys]\n    if not (1<=len(keys)<=2 and (len(keys)==1 or \'customer_id\' in keys)):\n        raise ValueError(\'Return one customer/article/category key or customer_id + article/category; no channel/age/price keys\')\n    if not features:\n        raise ValueError(\'Return at least one numeric f_* feature in addition to keys\')\n    for name in features:\n        if not re.fullmatch(r\'f_[a-z][a-z0-9_]{0,45}\',name):\n            raise ValueError(\'Output feature names must be unique f_lowercase_names: \'+name)\n    for name,dtype,*_ in description:\n        if not NUMERIC_TYPE.fullmatch(dtype):\n            raise ValueError(\'Output must be numeric, including keys; \'+name+\' has type \'+dtype)\n    return keys,features\n\n\ndef validate_program(program):\n    if not isinstance(program,dict) or set(program)!={\'queries\'} or not isinstance(program[\'queries\'],list):\n        raise ValueError(\'Host program must contain queries only\')\n    if len(program[\'queries\'])>MAX_QUERIES:\n        raise ValueError(\'At most6 complete SELECT queries\')\n    if not program[\'queries\']: return program  # host-only raw control\n    con=duckdb.connect(config={\'enable_external_access\':\'false\',\'threads\':\'1\',\'memory_limit\':\'128MB\'})\n    con.execute(\'CREATE TABLE history (\'+\',\'.join(n+(\' DOUBLE\' if n in [\'price\',\'customer_age\'] else \' BIGINT\') for n in HISTORY)+\')\')\n    names,issues=set(),[]\n    try:\n        for i,query in enumerate(program[\'queries\'],1):\n            try:\n                if not isinstance(query,str) or len(query)>24000: raise ValueError(\'Query must be SQL text <=24000characters\')\n                validate_ast(query)\n                keys,features=schema(con,query)\n                overlap=names.intersection(features)\n                if overlap: raise ValueError(\'Feature names repeated across queries: \'+\',\'.join(sorted(overlap)))\n                names.update(features)\n            except Exception as error:\n                issues.append(dict(query=i,sql=query,reason=str(error)[:650],\n                                   hint=\'Use the documented history columns; keys+numeric f_* outputs; full GROUP BY. Quantile: quantile_cont(price,0.5). CTEs may compute ratios before final SELECT.\'))\n        if len(names)>MAX_FEATURES:\n            issues.append(dict(query=0,sql=\'\',reason=\'More than36 output features\',hint=\'Keep the complete set compact\'))\n        if issues: raise SQLProgramError(issues)\n    finally:\n        con.close()\n    return program\n\n\ndef parse(text):\n    source=extract_sql(text)\n    # DuckDB preserves original SQL, including CTEs/window syntax. SQLGlot only\n    # validates and fingerprints; it never rewrites the executed program.\n    con=duckdb.connect()\n    try:\n        statements=con.extract_statements(source)\n        queries=[statement.query.strip().rstrip(\';\') for statement in statements]\n    except duckdb.Error as error:\n        raise ValueError(str(error)) from None\n    finally: con.close()\n    if not queries: raise ValueError(\'Raw control is host-only: propose at least one feature SELECT\')\n    return validate_program({\'queries\':queries})\n\n\ndef render(program):\n    return \';\\n\\n\'.join(program[\'queries\'])+\';\' if program[\'queries\'] else \'-- Raw15 control: no engineered feature query\'\n\n\ndef program_key(program):\n    canonical=[]\n    for query in program[\'queries\']:\n        tree=sqlglot.parse_one(query,read=\'duckdb\')\n        names={}\n        for n in tree.find_all(exp.Identifier):\n            value=n.name.lower()\n            if value.startswith(\'f_\'):\n                if value not in names: names[value]=\'f_normalized_\'+str(len(names))\n                n.set(\'this\',names[value])\n        # Conservative AST dedup: not general SQL/algebraic equivalence.\n        canonical.append(tree.sql(dialect=\'duckdb\',normalize=True,pretty=False,comments=False))\n    return hashlib.sha256(json.dumps(sorted(canonical)).encode()).hexdigest()\n\n\ndef signature(text):\n    try: return program_key(parse(text))\n    except Exception: return re.sub(r\'\\s+\',\' \',text.strip()).lower()\n\n\nclass FeatureEngine(BaseEngine):\n    def __init__(self,scratch,data_dir=None):\n        super().__init__(scratch,data_dir)\n        self.query_con=None\n\n    def prepare(self,timestamp,users,program):\n        validate_program(program)\n        super().prepare(timestamp,users,{\'aggregates\':[],\'derived\':[]})\n        if self.query_con is not None: self.query_con.close()\n        # The LLM connection contains PHYSICALLY PAST-ONLY history. No tx,\n        # labels, candidate rows, targets, or future transactions are registered.\n        past=self.con.execute(\'SELECT * FROM history\').fetch_arrow_table()\n        self.query_con=duckdb.connect(config={\'enable_external_access\':\'false\',\'threads\':\'2\',\'memory_limit\':\'768MB\'})\n        self.query_con.register(\'history\',past)\n        all_names=[]\n        for index,query in enumerate(program[\'queries\']):\n            timer=threading.Timer(60,self.query_con.interrupt)\n            timer.daemon=True\n            timer.start()\n            try:\n                keys,names=schema(self.query_con,query)\n                # Bound materialization; reject oversize results, never accept a\n                # silently truncated feature table.\n                result=self.query_con.execute(\'SELECT * FROM (\\n\'+query+\'\\n) LIMIT 250001\').fetch_df()\n            except Exception as error:\n                raise ValueError(f\'query{index+1}: {error}\') from None\n            finally: timer.cancel()\n            if len(result)>250000:\n                raise ValueError(f\'query{index+1}: too many output rows; aggregate by your join keys\')\n            if result.duplicated(subset=keys).any():\n                raise ValueError(f\'query{index+1}: duplicate key rows would multiply candidate pairs; return exactly one row per \'+str(keys))\n            for key in keys:\n                allowed=np.asarray(past.column(key).to_pandas().dropna().unique())\n                values=result[key].dropna().to_numpy()\n                if not np.isin(values,allowed).all():\n                    raise ValueError(f\'query{index+1}: invented/transformed join key values in \'+key)\n            table=\'full_sql_group_\'+str(index)\n            self.con.register(\'full_sql_result\',result)\n            self.con.execute(f\'CREATE OR REPLACE TEMP TABLE {table} AS SELECT * FROM full_sql_result\')\n            self.con.unregister(\'full_sql_result\')\n            entries=[{\'name\':n} for n in names]\n            self.groups.append((table,keys,entries))\n            all_names.extend(names)\n        # Reuse the frozen host join/matrix implementation, with no derived SQL\n        # stage: all formulas were already computed by the LLM\'s complete SELECTs.\n        self.program={\'aggregates\':[{\'name\':n} for n in all_names],\'derived\':[]}\n\n    def close(self):\n        if self.query_con is not None: self.query_con.close()\n        super().close()\n', 'prompting.py': '"""The model returns complete SQL statements, not the former JSON DSL."""\nfrom full_sql import HISTORY, render\n\nINSTRUCTION = \'\'\'Write a COMPLETE DuckDB SQL feature-engineering script for H&M recommendations.\nReturn ONLY SQL, no JSON/Python/prose. Separate complete SELECT statements with semicolons.\nWITH ... SELECT CTEs are welcome. Each query is independent: it can see ONLY history and its own CTEs.\nDo not CREATE tables/views: the host executes your SELECTs and handles final candidate joins.\nPredict purchases in the NEXT7days; fitness is LightGBM validation MAP@12, not training loss.\n\nhistory contains only purchases at/before cutoff, from the frozen5000-customer cohort, not full H&M.\nColumns: {columns}.\nprice=normalized purchase price; days_ago=days before cutoff; sales_channel_id=1/2.\nThe clothing-category columns describe the PURCHASED article; customer_age is numeric age.\nThere is no gender, raw_*, timestamp, t_dat, label, target, or future data table in this database.\nAll historical rows are available; time windows use WHERE/FILTER on days_ago.\n\nEach SELECT returns join keys and numeric features named f_lowercase_name.\nSupported final keys: customer_id; article_id; ONE clothing category; customer_id+article_id;\nor customer_id+ONE clothing category. Return EXACTLY ONE ROW per chosen key combination.\nPreserve key names/values. No sales_channel_id/price/age/days_ago join keys. Do not output IDs as features.\nAll f_* output names must be unique across all queries. At most6 SELECTs and36 output features total.\nStart compact, about2-3queries and6-10features. Return a complete set, not a patch.\nIntermediate CTE columns may have descriptive names. Ratios/transformations belong in your SELECTs.\nThe host appends the SAME15 raw customer/article fields as previous experiments automatically.\nOnly past history goes into your SQL; no labels, candidate rows, other files, tables, or external functions.\nCommon numeric SQL: count/sum/avg/min/max/stddev_pop/median/quantile_cont, approx_count_distinct,\nmode, coalesce/nullif, abs/sqrt/ln/log/power, greatest/least, round/floor/ceil, cast, CASE WHEN.\nFor age bins return numbers0/1/2, not strings. Use nullif(denominator,0) for ratios.\nQuantile syntax: quantile_cont(price,0.5). Mode: mode(customer_age ORDER BY customer_age).\nSELECT aliases, CTEs, joins within history-derived CTEs, and numeric window functions are allowed.\nWindow outputs still must yield one row per final key. Use deterministic ordering, not random().\nHistory is purchases, not views; sum(price) is spending, not discount; recent recency=min(days_ago).\n\nSYNTAX EXAMPLES ONLY, choose your own complete useful feature set:\nWITH counts AS (\n  SELECT customer_id, count(*) AS total_count,\n         count(*) FILTER(WHERE days_ago<=7) AS recent_count\n  FROM history GROUP BY customer_id\n)\nSELECT customer_id, total_count AS f_total_count, recent_count AS f_recent_count,\n       recent_count/nullif(total_count,0) AS f_recent_share FROM counts;\n\nSELECT article_id, count(*) FILTER(WHERE days_ago<=7) AS f_item_recent_count,\n       avg(price) AS f_item_mean_price FROM history GROUP BY article_id;\n\'\'\'.replace(\'{columns}\', \', \'.join(HISTORY))\n\n\ndef make_prompt(data,best,previous,config):\n    prompt=INSTRUCTION+f\'\'\'\nFIXED COMPARISON: {config[\'train_pairs\']} unchanged training pairs/4804 user-times/8 snapshots.\nLightGBM150rounds/31leaves/4threads. Search128users × ALL5000items.\nTop3search+raw compete on disjoint confirmation88. Heldout88/test250 not used for selection.\nEntire response <=2048tokens. Previous best complete SQL in YOUR island,\nsearch MAP fraction={best[\'score\']}:\n\'\'\'+render(best[\'program\'])\n    # These examples are static fields from an early training snapshot, not labels.\n    prompt+=\'\\nRaw training candidate examples (appended by HOST, unavailable inside history SQL):\\n\'+config[\'training_examples\']\n    if previous:\n        prompt+=\'\\nREPAIR: Return the COMPLETE corrected SQL script; fix all listed queries.\\nPrevious SQL response:\\n\'+previous[\'text\']\n        prompt+=\'\\nRejection diagnostics:\\n\'+previous[\'error\']\n        prompt+=\'\\nFor duplicates change statistics/windows/groupings, not just feature/CTE names. No JSON.\'\n    return prompt\n', 'runtime.py': '"""Matched local-Qwen SQL protocol for two sequential Kaggle LLMs."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom worker import ap12, sha, write\nfrom features import FeatureEngine, validate_program\nfrom full_sql import parse, render, program_key, signature, SQLProgramError as DiagnosticError\nfrom prompting import make_prompt\n\n\ndef locate_bundle(input_root, working, expected):\n    matches = []\n    for path in Path(input_root).rglob(\'manifest.json\'):\n        try:\n            if json.loads(path.read_text()).get(\'bundle_id\') == \'hm_small_sql_compare_v1\':\n                matches.append(path.parent)\n        except (ValueError, OSError):\n            pass\n    if not matches:\n        archives = list(Path(input_root).rglob(\'hm_small_sql_compare_v1.zip\'))\n        if len(archives) != 1:\n            raise FileNotFoundError(\'Attach hm_small_sql_compare_v1.zip as a Kaggle Dataset/Input\')\n        destination = Path(working) / \'hm_small_sql_compare_input\'\n        destination.mkdir(exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as archive:\n            for member in archive.infolist():\n                if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):\n                    raise ValueError(\'Unsafe archive member\')\n            archive.extractall(destination)\n        matches = [destination]\n    if len(matches) != 1:\n        raise ValueError(\'Attach exactly one small comparison bundle\')\n    root = matches[0]\n    assert sha(root / \'manifest.json\') == expected, \'Unexpected bundle version\'\n    manifest = json.loads((root / \'manifest.json\').read_text())\n    for name, digest in manifest[\'files\'].items():\n        assert sha(root / name) == digest, \'Checksum mismatch: \' + name\n    return root, manifest\n\n\ndef archive_run(out):\n    out = Path(out)\n    temporary = out.parent / (out.name + \'_results.partial\')\n    with zipfile.ZipFile(temporary, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file() and \'scratch\' not in p.parts:\n                archive.write(p, str(p.relative_to(out)))\n    final = out.parent / (out.name + \'_results.zip\')\n    temporary.replace(final)\n    return final\n\n\ndef evaluate(data, directory, stage, config_path, replay=False):\n    directory = Path(directory)\n    result = directory / f\'{stage}_metrics.json\'\n    if result.exists():\n        metric = json.loads(result.read_text())\n        if stage != \'preflight\':\n            assert metric[\'program_sha256\'] == sha(directory / \'program.json\')\n            assert metric[\'model_sha256\'] == sha(directory / \'model.txt\')\n        return metric\n    config = json.loads(Path(config_path).read_text())\n    env = {k: v for k, v in os.environ.items() if not any(s in k.upper() for s in [\'TOKEN\', \'SECRET\', \'API_KEY\'])}\n    env.update(CUDA_VISIBLE_DEVICES=\'\', OMP_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\')\n    command = [sys.executable, \'-u\', str(Path(__file__).with_name(\'worker.py\')), \'--data\', str(data),\n               \'--directory\', str(directory), \'--stage\', stage, \'--config\', str(config_path)]\n    with (directory / f\'{stage}.log\').open(\'w\') as log:\n        completed = subprocess.run(command, env=env, stdout=log, stderr=subprocess.STDOUT,\n                                   timeout=config[\'evaluation_timeout_seconds\'], check=False)\n    if completed.returncode or not result.exists():\n        raise RuntimeError((directory / f\'{stage}.log\').read_text()[-3500:])\n    return json.loads(result.read_text())\n\n\n\n\n\n\n\n\ndef progress(out, trials, config):\n    valid = [t for t in trials if t[\'status\'] == \'valid\']\n    state = {\'calls\': len(trials), \'measured\': len(valid), \'target\': config[\'target_successful_distinct\'],\n             \'max_llm_calls\': config[\'max_llm_calls\'], \'trials\': trials}\n    write(Path(out) / \'search_state.json\', state)\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    if valid:\n        calls = [t[\'call\'] for t in valid]\n        scores = np.array([t[\'score\'] for t in valid]) * 100\n        axes[0].scatter(calls, scores, c=[t[\'island\'] for t in valid], cmap=\'viridis\', vmin=0, vmax=2)\n        axes[0].plot(calls, np.maximum.accumulate(scores), color=\'black\', label=\'Best search MAP\')\n        axes[0].legend()\n    axes[0].set(xlabel=\'LLM call\', ylabel=\'Search MAP@12 (%)\', title=\'Search only; color = island\')\n    counts = pd.Series([t[\'status\'] + \':\' + t.get(\'failure_stage\', \'\') for t in trials]).value_counts()\n    axes[1].barh(counts.index, counts.values)\n    axes[1].set(title=\'Accepted / rejected / duplicate\', xlabel=\'Calls\')\n    fig.tight_layout()\n    fig.savefig(Path(out) / \'search_progress.png\', dpi=140)\n    plt.close(fig)\n    return valid\n\n\ndef search(data, out, generate, config_path):\n    """Replay stored trials on resume; never request generation twice for a saved response."""\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    config = json.loads(Path(config_path).read_text())\n    if (out / \'selected.json\').exists():\n        return json.loads((out / \'selected.json\').read_text())\n    raw = out.parent / \'raw_control\'\n    baseline = evaluate(data, raw, \'search\', config_path)\n    memories = [[{\'directory\': str(raw), \'score\': baseline[\'map\'], \'program\': {\'queries\': []}}] for _ in range(3)]\n    trials, previous, repair, proposal = [], None, 0, 0\n    seen = set()\n    while sum(t[\'status\'] == \'valid\' for t in trials) < config[\'target_successful_distinct\'] and len(trials) < config[\'max_llm_calls\']:\n        number = len(trials) + 1\n        island = proposal % 3\n        directory = out / \'candidates\' / f\'candidate_{number:03d}\'\n        directory.mkdir(parents=True, exist_ok=True)\n        trial_path = directory / \'trial.json\'\n        if trial_path.exists():\n            trial = json.loads(trial_path.read_text())\n        else:\n            trial = {\'call\': number, \'proposal\': proposal, \'island\': island, \'repair\': repair,\n                     \'parent_call\': previous[\'call\'] if previous else None, \'directory\': str(directory)}\n            stage = \'generation\'\n            tick = time.perf_counter()\n            text = \'\'\n            try:\n                best = max(memories[island], key=lambda x: x[\'score\'])\n                prompt = make_prompt(data, best, previous, config)\n                (directory / \'prompt.txt\').write_text(prompt, encoding=\'utf-8\')\n                response_path = directory / \'response.json\'\n                print(f"{out.name}: call{number}/{config[\'max_llm_calls\']}, measured{len(seen)}/{config[\'target_successful_distinct\']}, island{island}, repair{repair}/{config[\'max_repairs_per_proposal\']}", flush=True)\n                if response_path.exists():\n                    response = json.loads(response_path.read_text())\n                else:\n                    response = generate(prompt, 42 + number - 1)\n                    write(response_path, response)\n                text = response[\'text\']\n                trial[\'response_signature\'] = signature(text)\n                trial[\'unchanged_repair\'] = bool(previous and trial[\'response_signature\'] and trial[\'response_signature\'] == previous.get(\'response_signature\'))\n                trial.update({k: v for k, v in response.items() if k != \'text\'})\n                stage = \'parse_and_validate\'\n                if response.get(\'truncated\'):\n                    raise ValueError(\'Output truncated; return a smaller complete feature program\')\n                program = parse(text)\n                key = program_key(program)\n                trial[\'program_key\'] = key\n                if previous and key == previous.get(\'program_key\'):\n                    trial[\'unchanged_repair\'] = True\n                write(directory / \'program.json\', program)\n                (directory / \'program.sql\').write_text(render(program), encoding=\'utf-8\')\n                if key in seen:\n                    matched = next(t for t in trials if t[\'status\'] == \'valid\' and t[\'program_key\'] == key)\n                    error = \'Already evaluated feature set; choose a distinct complete set\'\n                    if config.get(\'arm\') == \'qwen_full_sql\':\n                        error += (\'; duplicate of call\' + str(matched[\'call\']) + \'. \'\n                                  + \'Your Previous SQL response already contains the complete matching program.\'\n                                  + \'\\nChange a meaningful axis: time window, statistic, or candidate grouping. \'\n                                  + \'For example add/change customer+article/category history. Renaming is not a new set.\')\n                    trial.update(status=\'duplicate\', error=error, failure_stage=\'deduplication\', duplicate_of=matched[\'call\'])\n                else:\n                    stage = \'preflight\'\n                    evaluate(data, directory, \'preflight\', config_path)\n                    print(f\'call{number}: preflight passed; LightGBM training on72813 pairs\', flush=True)\n                    stage = \'training_and_search\'\n                    score = evaluate(data, directory, \'search\', config_path)\n                    trial.update(status=\'valid\', score=score[\'map\'], program=program)\n            except Exception as error:\n                if isinstance(error, DiagnosticError):\n                    write(directory / \'diagnostics.json\', error.issues)\n                    trial[\'diagnostic_queries\'] = len(error.issues)\n                trial.update(status=\'rejected\', failure_stage=stage, error=str(error))\n                (directory / \'error.txt\').write_text(traceback.format_exc())\n            trial.update(elapsed_seconds=time.perf_counter() - tick)\n            write(trial_path, trial)\n        trials.append(trial)\n        if trial[\'status\'] == \'valid\':\n            seen.add(trial[\'program_key\'])\n            memories[island].append(trial)\n            print(f"ACCEPTED call{number}: MAP@12={100*trial[\'score\']:.6f}%", flush=True)\n        else:\n            print(f"{trial[\'status\'].upper()} call{number}: {trial[\'error\']}", flush=True)\n        progress(out, trials, config)\n        archive_run(out.parent)\n        stop_unchanged = config.get(\'arm\') == \'qwen_full_sql\' and trial.get(\'unchanged_repair\', False)\n        if trial[\'status\'] != \'valid\' and repair < config[\'max_repairs_per_proposal\'] and not stop_unchanged:\n            text = json.loads((directory / \'response.json\').read_text()).get(\'text\', \'\') if (directory / \'response.json\').exists() else \'\'\n            previous = {**trial, \'text\': text}\n            repair += 1\n        else:\n            previous, repair, proposal = None, 0, proposal + 1\n    valid = sorted([t for t in trials if t[\'status\'] == \'valid\'], key=lambda t: (-t[\'score\'], t[\'call\']))\n    candidates = [{\'directory\': str(raw), \'call\': None}] + valid[:3]\n    confirmation = []\n    for trial in candidates:\n        metric = evaluate(data, trial[\'directory\'], \'confirm\', config_path)\n        confirmation.append({\'directory\': trial[\'directory\'], \'call\': trial[\'call\'], \'confirmation_map\': metric[\'map\']})\n    # Stable ties prefer raw, then earlier higher-search candidate. No test selection.\n    winner = max(confirmation, key=lambda t: t[\'confirmation_map\'])\n    selected = {**winner, \'selection\': \'top3 search plus raw -> disjoint confirmation88\',\n                \'program_sha256\': sha(Path(winner[\'directory\']) / \'program.json\'),\n                \'model_sha256\': sha(Path(winner[\'directory\']) / \'model.txt\'),\n                \'successful_distinct\': len(valid), \'budget_complete\': len(valid) == config[\'target_successful_distinct\'],\n                \'llm_calls\': len(trials), \'confirmation\': confirmation}\n    write(out / \'selected.json\', selected)\n    archive_run(out.parent)\n    return selected\n\n\nclass Generator:\n    def __init__(self, spec, config, token=None, cache_dir=None):\n        import torch\n        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Enable a Kaggle T4 GPU\')\n        self.config = config\n        self.torch = torch\n        self.tokenizer = AutoTokenizer.from_pretrained(spec[\'model_id\'], revision=spec[\'revision\'], token=token, cache_dir=cache_dir)\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n        self.model = AutoModelForCausalLM.from_pretrained(spec[\'model_id\'], revision=spec[\'revision\'], token=token,\n                     quantization_config=quant, torch_dtype=torch.float16, device_map={\'\': 0}, attn_implementation=\'sdpa\', cache_dir=cache_dir)\n        self.model.eval()\n        self.eos = self.model.generation_config.eos_token_id\n        self.pad = self.tokenizer.eos_token_id\n\n    def __call__(self, prompt, seed):\n        from transformers import set_seed\n        set_seed(seed)\n        messages = [{\'role\': \'system\', \'content\': \'Return only a complete DuckDB SQL script of SELECT/WITH queries following the user interface.\'},\n                    {\'role\': \'user\', \'content\': prompt}]\n        inputs = self.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,\n                                                    return_dict=True, return_tensors=\'pt\').to(\'cuda:0\')\n        tokens = inputs[\'input_ids\'].shape[1]\n        if tokens > self.config[\'max_input_tokens\']:\n            raise ValueError(f\'Prompt too long: {tokens}; no silent truncation\')\n        started = time.perf_counter()\n        with self.torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=self.config[\'max_new_tokens\'], do_sample=True,\n                       temperature=self.config[\'temperature\'], top_p=1., top_k=0, repetition_penalty=1.,\n                       eos_token_id=self.eos, pad_token_id=self.pad)\n        generated = output[0, tokens:]\n        eos = self.eos if isinstance(self.eos, list) else [self.eos]\n        return {\'text\': self.tokenizer.decode(generated, skip_special_tokens=True), \'seed\': seed,\n                \'prompt_tokens\': tokens, \'generated_tokens\': len(generated),\n                \'seconds\': time.perf_counter() - started,\n                \'truncated\': len(generated) >= self.config[\'max_new_tokens\'] and int(generated[-1]) not in eos}\n\n    def close(self):\n        del self.model, self.tokenizer\n        gc.collect()\n        self.torch.cuda.empty_cache()\n\n\ndef finish(data, out, config_path):\n    """Open final targets/references only after all full SQL selections are frozen."""\n    data, out = Path(data), Path(out)\n    config = json.loads(Path(config_path).read_text())\n    smoke = config.get(\'validation_only_fake_callbacks\', False)\n    for name in config[\'arms\']:\n        assert (out / name / \'selected.json\').exists(), \'All full SQL arm must be selected before test\'\n    rows, aps = [], {}\n    programs = {\'raw_control\': out / \'raw_control\'}\n    for name in config[\'arms\']:\n        selected = json.loads((out / name / \'selected.json\').read_text())\n        assert sha(Path(selected[\'directory\']) / \'program.json\') == selected[\'program_sha256\']\n        assert sha(Path(selected[\'directory\']) / \'model.txt\') == selected[\'model_sha256\']\n        programs[name + \'_LightGBM\'] = Path(selected[\'directory\'])\n    for name, directory in programs.items():\n        row = {\'model\': name}\n        for stage in [\'heldout_val\', \'val\', \'test\']:\n            metric = evaluate(data, directory, stage, config_path)\n            replay = evaluate(data, directory, \'replay_\' + stage, config_path)\n            assert replay[\'passed\']\n            row[stage + \'_map_percent\'] = 100 * metric[\'map\']\n            if stage == \'test\':\n                aps[name] = np.load(directory / \'test_ap.npy\')\n        rows.append(row)\n    selection = json.loads((data / \'selection.json\').read_text())\n    sources = json.loads((data / \'references/sources.json\').read_text())\n    for source in sources:\n        name = source[\'model\']\n        row = {\'model\': name}\n        for split in [\'val\', \'test\']:\n            target = pd.read_parquet(data / f\'{split}_targets.parquet\')\n            pred = np.load(data / f\'references/{name}_{split}.npy\')\n            ap = ap12(pred, target.article_id)\n            expected = source[\'metrics\'][f\'{split}_metrics\'][\'map\']\n            assert abs(float(ap.mean()) - expected) < 1e-10, \'Reference alignment/metric mismatch: \' + name\n            row[split + \'_map_percent\'] = 100 * float(ap.mean())\n            if split == \'val\':\n                row[\'heldout_val_map_percent\'] = 100 * float(ap[selection[\'heldout_val\']].mean())\n            else:\n                aps[name] = ap\n        rows.append(row)\n    # Same time-safe popularity control used by the small ContextGNN report.\n    tx = pd.read_parquet(data / \'transactions.parquet\')\n    row = {\'model\': \'popularity\'}\n    for split in [\'val\', \'test\']:\n        target = pd.read_parquet(data / f\'{split}_targets.parquet\')\n        counts = tx[tx.t_dat <= target.timestamp.iloc[0]].article_id.value_counts().reindex(np.arange(5000), fill_value=0)\n        ranking = np.lexsort((np.arange(5000), -counts.to_numpy()))[:12]\n        ap = ap12(np.tile(ranking, (len(target), 1)), target.article_id)\n        row[split + \'_map_percent\'] = 100 * float(ap.mean())\n        if split == \'val\':\n            row[\'heldout_val_map_percent\'] = 100 * float(ap[selection[\'heldout_val\']].mean())\n        else:\n            aps[\'popularity\'] = ap\n    rows.append(row)\n    frame = pd.DataFrame(rows)\n    frame.to_csv(out / \'comparison.csv\', index=False)\n    fig, axes = plt.subplots(1, 2, figsize=(15, 6))\n    for axis, split in zip(axes, [\'val\', \'test\']):\n        axis.barh(frame.model, frame[split + \'_map_percent\'])\n        axis.set(title=split + \' — same small catalog5000\', xlabel=\'MAP@12 (%)\')\n        axis.grid(axis=\'x\', alpha=.2)\n    if smoke:\n        fig.suptitle(\'CPU SMOKE — scripted SQL responses; not LLM performance\', color=\'darkred\')\n    fig.tight_layout()\n    fig.savefig(out / \'comparison.png\', dpi=160)\n    plt.close(fig)\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n    for name in [n + \'_LightGBM\' for n in config[\'arms\']] + [\'raw_control\', \'contextgnn\', \'lightgbm_manual\']:\n        values = aps[name]\n        axes[0].hist(values[values > 0] * 100, bins=np.linspace(0, 100, 21), histtype=\'step\', label=name)\n        axes[1].bar(name, 100 * np.mean(values == 0))\n    axes[0].set(xlabel=\'Per-user AP@12 (%) among AP>0\', ylabel=\'Users\', title=\'Test distribution\')\n    axes[0].legend(fontsize=7)\n    axes[1].set(ylabel=\'Users with AP@12=0 (%)\', ylim=(0, 100))\n    axes[1].tick_params(axis=\'x\', labelrotation=35)\n    if smoke:\n        fig.suptitle(\'CPU SMOKE — scripted SQL responses\', color=\'darkred\')\n    fig.tight_layout()\n    fig.savefig(out / \'test_histograms.png\', dpi=160)\n    plt.close(fig)\n    fig, axis = plt.subplots(figsize=(8, 4))\n    for name, directory in programs.items():\n        curve = json.loads((directory / \'learning_curve.json\').read_text())[\'train\'][\'binary_logloss\']\n        axis.plot(np.arange(1, len(curve) + 1), curve, label=name)\n    axis.set(xlabel=\'LightGBM round\', ylabel=\'Training binary logloss\', title=\'Loss is diagnostic; selection uses MAP@12\')\n    axis.legend()\n    fig.tight_layout()\n    fig.savefig(out / \'lightgbm_training.png\', dpi=160)\n    plt.close(fig)\n    diff = aps[config[\'arms\'][0] + \'_LightGBM\'] - aps[\'raw_control\']\n    rng = np.random.default_rng(42)\n    bootstrap = diff[rng.integers(0, len(diff), size=(10000, len(diff)))].mean(1) * 100\n    write(out / \'paired_bootstrap.json\', {\'first_arm_minus_raw_pp\': float(diff.mean() * 100),\n          \'ci95_pp\': np.quantile(bootstrap, [.025, .975]).tolist(), \'test_users\': len(diff), \'seed\': 42,\n          \'note\': \'Exploratory paired-user interval, not multi-seed uncertainty\'})\n    lines = [\'# Qwen7B full SQL — small H&M SQL feature search\', \'\',\n             \'Same frozen small-v1 data,72813 train pairs,304val/250test users,all5000 catalog items.\',\n             \'Qwen7B returns complete SELECT scripts; NF4/FP16 loading matches the control.\',\n             \'Test250 has been inspected in earlier experiments: this is a repeated diagnostic evaluation, not a new untouched test.\',\n             \'Old local Qwen used Q4_K_M/Ollama: this is not byte-identical quantization.\',\n             \'All selections frozen before final test. Same20-success target/300-call cap/3repairs/3islands.\',\n             \'Validation304 includes search/confirmation rows; heldout88 is separate for LLMs.\',\n             \'References selected on original full validation, so their heldout88 is not an untouched selection set.\',\n             \'Baselines have different model tuning/training/pretraining budgets. One seed;250test users.\', \'\',\n             \'| Model | Val MAP,% | Test MAP,% |\', \'|---|---:|---:|\']\n    if smoke:\n        lines.insert(1, \'**CPU SMOKE ONLY: scripted SQL responses, not actual LLM performance.**\')\n    for record in rows:\n        lines.append(f"| {record[\'model\']} | {record[\'val_map_percent\']:.5f} | {record[\'test_map_percent\']:.5f} |")\n    for name in config[\'arms\']:\n        selected = json.loads((out / name / \'selected.json\').read_text())\n        lines += [\'\', f"{name}: {selected[\'successful_distinct\']} successful /{selected[\'llm_calls\']} calls; budget_complete={selected[\'budget_complete\']}; selected_call={selected[\'call\']} (null=raw control)."]\n    lines += [\'\', \'![Comparison](comparison.png)\', \'\', \'![Test AP histograms](test_histograms.png)\']\n    (out / \'report.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    budgets_complete = all(json.loads((out / name / \'selected.json\').read_text())[\'budget_complete\'] for name in config[\'arms\'])\n    write(out / \'complete.json\', {\'status\': \'complete\' if budgets_complete else \'complete_with_incomplete_search\',\n                                \'search_budgets_complete\': budgets_complete, \'independent_frozen_replays\': True,\n                                \'references_recomputed_from_predictions\': True})\n    efficiency_report(out, config)\n    archive_run(out)\n    return frame\n\n\n"""Generation efficiency is separate from recommendation quality."""\ndef efficiency_report(out, config):\n    rows = []\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4))\n    for arm in config[\'arms\']:\n        trials = json.loads((out / arm / \'search_state.json\').read_text())[\'trials\']\n        row = {\'arm\': arm, \'calls\': len(trials)}\n        row.update({s: sum(t[\'status\'] == s for t in trials) for s in [\'valid\', \'rejected\', \'duplicate\']})\n        row.update(repair_calls=sum(t[\'repair\'] > 0 for t in trials),\n                   unchanged_repairs=sum(t.get(\'unchanged_repair\', False) for t in trials),\n                   generated_tokens=sum(t.get(\'generated_tokens\', 0) for t in trials),\n                   generation_seconds=sum(t.get(\'seconds\', 0) for t in trials),\n                   evaluation_and_generation_seconds=sum(t.get(\'elapsed_seconds\', 0) for t in trials),\n                   target_reached=row[\'valid\'] == config[\'target_successful_distinct\'])\n        rows.append(row)\n        valid = [t for t in trials if t[\'status\'] == \'valid\']\n        axes[0].step([0] + [t[\'call\'] for t in valid], [0] + list(range(1, len(valid) + 1)), where=\'post\', label=arm)\n    frame = pd.DataFrame(rows)\n    frame.to_csv(out / \'generation_efficiency.csv\', index=False)\n    axes[0].set(xlabel=\'LLM calls\', ylabel=\'Distinct measured sets\', title=\'Calls to measured sets\')\n    axes[0].legend()\n    frame.set_index(\'arm\')[[\'valid\', \'rejected\', \'duplicate\']].plot.bar(stacked=True, ax=axes[1])\n    axes[1].set(title=\'All calls counted\', ylabel=\'Calls\')\n    axes[2].bar(frame.arm, frame.generation_seconds / 60)\n    axes[2].set(title=\'Generation time\', ylabel=\'Minutes\')\n    if config.get(\'validation_only_fake_callbacks\'):\n        fig.suptitle(\'CPU SMOKE: scripted responses, not LLM performance\', color=\'darkred\')\n    fig.tight_layout()\n    fig.savefig(out / \'generation_efficiency.png\', dpi=160)\n    plt.close(fig)\n    with (out / \'report.md\').open(\'a\', encoding=\'utf-8\') as stream:\n        stream.write(\'\\n\\nFull SQL: ordinary SELECT/WITH scripts with host-owned candidate joins.\\n\'\n                     \'Compare locally to archived Qwen7B and H003; SQL interface/dedup semantics differ.\\n\'\n                     \'The same previously inspected test is reused diagnostically, never to select variants.\\n\'\n                     \'See generation_efficiency.csv and generation_efficiency.png.\\n\')\n\n'}
MODULE_SHA = {'base_features.py': 'd4625f7cfe65395925a54984835a5b9536dbde586be7916deaaacbe50069efe1', 'worker.py': '3844f32b7a86a387725f6a5c1cc0121c8f7921aee2893e17847bd6c0c47c92a4', 'features.py': '6aeefafe74853b1afe7d9bcacd184e6ab8725dc4f32b6a0941b92a63f7a0f628', 'full_sql.py': 'b7e0aa3e04d398b53fbe8067cf37569e575d2e6cb272c71be7d63fc5b9b3927c', 'prompting.py': 'c86ca707a7ba54b5922500f310939050ef8e0573737242d07e715b3b47243ec0', 'runtime.py': '7d361ad793955c084230562aad3eb1a4c2473ccd1803849970eb2664682b8b03'}
EXPECTED_MANIFEST = '3cb473dfafa1d60c4922fe85e63ae5b19efe0cf76f72cfd04164030132db01a3'
CODE = WORK / 'hm_small_full_sql_runtime_v2'
CODE.mkdir(exist_ok=True)
for name, source in MODULES.items():
    (CODE / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE))
import runtime as experiment
DATA, MANIFEST = experiment.locate_bundle('/kaggle/input', WORK, EXPECTED_MANIFEST)
print('Bundle:', MANIFEST['bundle_id'], '| train pairs:', MANIFEST['train_pairs'], '| catalog:', MANIFEST['catalog'])


In [ ]:
# Проверяем доступ к Qwen7B до начала первого поиска. Токен не сохраняется в конфиг.
TOKEN = None  # Public Qwen, pinned revision matches the control
from huggingface_hub import HfApi, hf_hub_download
for label, spec in MODELS.items():
    info = HfApi().model_info(spec['model_id'], revision=spec['revision'], token=TOKEN)
    spec['revision'] = info.sha
    hf_hub_download(spec['model_id'], 'config.json', revision=spec['revision'], token=TOKEN,
                    cache_dir=str(CACHE_ROOT / label))
CONFIG.update(models=MODELS, bundle_manifest_sha256=EXPECTED_MANIFEST, runtime_sha256=MODULE_SHA,
              train_pairs=MANIFEST['train_pairs'], train_pair_sha256=MANIFEST['train_pair_sha256'])
CONFIG['versions'] = {name: importlib.metadata.version(name) for name in
    ['torch', 'transformers', 'bitsandbytes', 'accelerate', 'lightgbm', 'duckdb', 'numpy', 'pandas', 'pyarrow', 'sqlglot']}
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle settings'
CONFIG['gpu'] = torch.cuda.get_device_name(0)
engine = experiment.FeatureEngine(WORK / 'example_scratch', DATA)
try:
    pairs = pd.read_parquet(DATA / 'train_pairs.parquet')
    sample = pairs[pairs.timestamp == pairs.timestamp.min()].head(3)
    engine.prepare(sample.timestamp.iloc[0], sample.customer_id.values, {'queries': []})
    CONFIG['training_examples'] = engine.frame(sample).to_json(orient='records')
finally:
    engine.close()
fingerprint = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
OUT = WORK / (CONFIG['protocol'] + '_' + fingerprint)
OUT.mkdir(exist_ok=True)
CONFIG_PATH = OUT / 'run_config.json'
if CONFIG_PATH.exists():
    assert json.loads(CONFIG_PATH.read_text()) == CONFIG, 'Changed config requires another run ID'
experiment.write(CONFIG_PATH, CONFIG)
for name, source in MODULES.items():
    destination = OUT / 'runtime_sources' / name
    destination.parent.mkdir(exist_ok=True)
    destination.write_text(source, encoding='utf-8')
shutil.copy2(DATA / 'manifest.json', OUT / 'input_manifest.json')
shutil.copy2(DATA / 'selection.json', OUT / 'validation_selection.json')
print('Run:', OUT.name, '| GPU:', CONFIG['gpu'])
print(json.dumps(MODELS, indent=2))


In [ ]:
raw = OUT / 'raw_control'
raw.mkdir(exist_ok=True)
experiment.write(raw / 'program.json', {'queries': []})
(raw / 'program.sql').write_text('-- Raw15 control', encoding='utf-8')
try:
    baseline = experiment.evaluate(DATA, raw, 'search', CONFIG_PATH)
    print('Raw15 search MAP@12:', 100 * baseline['map'], '%')
finally:
    print('Checkpoint ZIP:', experiment.archive_run(OUT))


In [ ]:
# All searches/confirmation are completed before final test evaluation.
generator = None
try:
    pending = [arm for arm in ARMS if not (OUT / arm / 'selected.json').exists()]
    if pending:
        generator = experiment.Generator(MODELS['qwen'], CONFIG, token=TOKEN, cache_dir=CACHE_ROOT / 'qwen')
    for arm in ARMS:
        arm_config = {**CONFIG, 'arm': arm, 'arm_protocol': CONFIG['protocol'] + '_' + arm}
        arm_path = OUT / ('config_' + arm + '.json')
        experiment.write(arm_path, arm_config)
        if not (OUT / arm / 'selected.json').exists():
            experiment.search(DATA, OUT / arm, generator, arm_path)
        print(arm, 'selection frozen:', json.loads((OUT / arm / 'selected.json').read_text()))
except BaseException:
    (OUT / 'failure_search.txt').write_text(traceback.format_exc())
    raise
finally:
    if generator is not None:
        generator.close()
        del generator
    gc.collect()
    print('Checkpoint ZIP:', experiment.archive_run(OUT))


In [ ]:
try:
    comparison = experiment.finish(DATA, OUT, CONFIG_PATH)
    display(comparison)
    for name in ['comparison.png', 'test_histograms.png', 'lightgbm_training.png', 'generation_efficiency.png']:
        display(Image(filename=str(OUT / name)))
    for label in ARMS:
        display(Image(filename=str(OUT / label / 'search_progress.png')))
except BaseException:
    (OUT / 'failure_final.txt').write_text(traceback.format_exc())
    raise
finally:
    archive = experiment.archive_run(OUT)
    print('Results ZIP:', archive)
    display(FileLink(str(archive)))
